# 🧪 Test de Agentes — Agencia de Viajes Inteligente

Ejecuta cada agente de forma **aislada** para verificar que:
1. La herramienta de búsqueda web funciona (Serper API)
2. El agente produce output JSON válido
3. Los datos son coherentes con precios reales

Ejecuta las celdas en orden. Cada agente es independiente.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

from dotenv import load_dotenv

# Cargar entorno
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "tests" else Path.cwd()
load_dotenv(PROJECT_DIR / ".env")

# Asegurar que el repo root está en el path
REPO_ROOT = str(PROJECT_DIR.parent.parent)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from crewai import Agent, Crew, Process, Task
from src.lab4_tip_planner.tools import (
    search_flights, search_car_rental, search_activities,
    search_accommodation, search_accommodation_image,
    search_activity_image, web_search,
)

import os
print("OPENAI_API_KEY:", "✓" if os.getenv("OPENAI_API_KEY") else "✗")
print("SERPER_API_KEY:", "✓" if os.getenv("SERPER_API_KEY") else "✗")

In [ ]:
# Datos del viaje de prueba
TRIP = {
    "origin": "Barcelona",
    "destination": "Reykjavik",
    "date": "2026-09-07",
    "return_date": "2026-09-14",
    "adults": 2,
    "days": 7,
    "max_night": 120,
}

print(f"🧪 Viaje de prueba: {TRIP['origin']} → {TRIP['destination']}")
print(f"   {TRIP['date']} → {TRIP['return_date']} ({TRIP['days']} días)")
print(f"   {TRIP['adults']} adultos, max {TRIP['max_night']}€/noche")

In [ ]:
def run_agent_test(name, agent, task_description):
    """Ejecuta un agente aislado y muestra el resultado."""
    print(f"\n{'='*60}")
    print(f"🧪 {name}")
    print(f"{'='*60}\n")

    task = Task(
        description=task_description,
        expected_output="JSON válido con los resultados",
        agent=agent,
    )

    crew = Crew(
        agents=[agent],
        tasks=[task],
        process=Process.sequential,
        verbose=True,
    )

    result = crew.kickoff()
    raw = result.raw.strip()

    print(f"\n{'─'*60}")
    print(f"📋 RESULTADO ({len(raw)} chars):")
    print(f"{'─'*60}")
    print(raw[:1000])

    # Verificar JSON
    start = raw.find("{")
    end = raw.rfind("}") + 1
    if start >= 0 and end > start:
        try:
            data = json.loads(raw[start:end])
            print(f"\n✅ JSON válido")
            print(json.dumps(data, indent=2, ensure_ascii=False)[:800])
            return data
        except json.JSONDecodeError as e:
            print(f"\n⚠️ JSON inválido: {e}")
    else:
        print(f"\n⚠️ No se encontró JSON en la respuesta")

    return raw

---
## ✈️ Test 1: Agente de Vuelos

Busca vuelos reales en Skyscanner, Kayak, Google Flights via web search.

In [ ]:
# Test directo de la herramienta (sin agente)
print("📡 Test directo: search_flights tool\n")

tool_result = search_flights.run(
    departure=TRIP["origin"],
    destination=TRIP["destination"],
    date=TRIP["date"],
    return_date=TRIP["return_date"],
    adults=TRIP["adults"],
)

data = json.loads(tool_result)
print(f"Skyscanner URL: {data['skyscanner_url']}")
print(f"\nGoogle Flights results (primeros 300 chars):")
print(str(data.get('google_flights_results', ''))[:300])
print(f"\nSkyscanner results (primeros 300 chars):")
print(str(data.get('skyscanner_results', ''))[:300])

In [ ]:
# Test con el agente completo
flight_agent = Agent(
    role="Especialista en Vuelos Internacionales",
    goal=(
        f"Encontrar vuelos reales de {TRIP['origin']} a {TRIP['destination']}. "
        "Usar SOLO precios reales de la búsqueda web, nunca inventar."
    ),
    backstory=(
        "Agente veterano con 20 años de experiencia. "
        "Busca en Skyscanner, Kayak y Google Flights. Solo precios verificables."
    ),
    tools=[search_flights, web_search],
    verbose=True,
    allow_delegation=False,
)

flight_result = run_agent_test(
    "Agente de Vuelos",
    flight_agent,
    f"Busca vuelos de {TRIP['origin']} a {TRIP['destination']}.\n"
    f"Ida: {TRIP['date']}, Vuelta: {TRIP['return_date']}, {TRIP['adults']} adultos.\n"
    f"Solo vuelos directos.\n\n"
    "INSTRUCCIONES:\n"
    "1. Usa 'Skyscanner flight search' con los parámetros correctos.\n"
    "2. Propón 3 opciones con precios REALES de la búsqueda.\n"
    "3. Incluye skyscanner_url en cada opción.\n\n"
    "FORMATO: JSON con {\"options\": [{\"airline\": ..., \"price_per_person\": ..., "
    "\"skyscanner_url\": ...}]}"
)

---
## 🚗 Test 2: Agente de Transporte

Busca alquiler de coches reales via web search.

In [ ]:
# Test directo de la herramienta
print("📡 Test directo: search_car_rental tool\n")

tool_result = search_car_rental.run(
    destination=TRIP["destination"],
    pickup_date=TRIP["date"],
    return_date=TRIP["return_date"],
)

data = json.loads(tool_result)
print(f"Resultados (primeros 400 chars):")
print(str(data.get('web_search_results', ''))[:400])

In [ ]:
# Test con el agente completo
transport_agent = Agent(
    role="Coordinador de Transporte",
    goal=f"Buscar opciones reales de alquiler de coche en {TRIP['destination']}",
    backstory="Experto en alquiler de vehículos internacionales. Solo datos reales.",
    tools=[search_car_rental, web_search],
    verbose=True,
    allow_delegation=False,
)

transport_result = run_agent_test(
    "Agente de Transporte",
    transport_agent,
    f"Busca alquiler de coche en {TRIP['destination']} por {TRIP['days']} días.\n"
    f"Desde {TRIP['date']} hasta {TRIP['return_date']}.\n\n"
    "INSTRUCCIONES:\n"
    "1. Usa 'Car rental search' con destination, pickup_date y return_date.\n"
    "2. Propón 3 opciones con precios reales.\n\n"
    "FORMATO: JSON con {\"options\": [{\"company\": ..., \"category\": ..., "
    "\"price_per_day\": ..., \"price_total\": ...}]}"
)

---
## 🎯 Test 3: Agente de Actividades

Busca actividades reales con precios de GetYourGuide, Viator, etc.

In [ ]:
# Test directo de la herramienta
print("📡 Test directo: search_activities tool\n")

tool_result = search_activities.run(
    destination=TRIP["destination"],
    activity_type="excursiones naturaleza",
)

data = json.loads(tool_result)
print(f"Resultados (primeros 400 chars):")
print(str(data.get('web_search_results', ''))[:400])

In [ ]:
# Test con el agente completo
activities_agent = Agent(
    role="Planificador de Actividades",
    goal=f"Planificar actividades para {TRIP['days']} días en {TRIP['destination']}",
    backstory="Guía local que prioriza experiencias memorables sobre cantidad.",
    tools=[search_activities, search_activity_image, web_search],
    verbose=True,
    allow_delegation=False,
)

activities_result = run_agent_test(
    "Agente de Actividades",
    activities_agent,
    f"Planifica actividades para {TRIP['days']} días en {TRIP['destination']}.\n"
    f"{TRIP['adults']} adultos.\n\n"
    "INSTRUCCIONES:\n"
    "1. Usa 'Activities search' para buscar precios reales.\n"
    "2. Separa free_activities y paid_activities por día.\n"
    "3. Para cada una indica start_time y duration_minutes.\n\n"
    "FORMATO: JSON con {\"days\": [{\"day\": 1, \"title\": ..., "
    "\"free_activities\": [...], \"paid_activities\": [...]}]}"
)

---
## 🏠 Test 4: Agente de Alojamientos

Busca alojamientos reales en Airbnb con imágenes.

In [ ]:
# Test directo de las herramientas
print("📡 Test directo: search_accommodation tool\n")

tool_result = search_accommodation.run(
    destination=f"{TRIP['destination']} centro",
    max_price=TRIP["max_night"],
    guests=TRIP["adults"],
)

data = json.loads(tool_result)
print(f"Resultados (primeros 400 chars):")
print(str(data.get('web_search_results', ''))[:400])

print(f"\n\n📡 Test directo: search_accommodation_image tool\n")

img_result = search_accommodation_image.run(
    query=f"apartamento centro {TRIP['destination']}"
)
img_data = json.loads(img_result)
print(f"Image URL: {img_data.get('image_url', 'N/A')[:80]}")
print(f"Listing URL: {img_data.get('listing_url', 'N/A')[:80]}")

In [ ]:
# Test con el agente completo
accommodation_agent = Agent(
    role="Curador de Alojamientos",
    goal=f"Buscar Airbnb en {TRIP['destination']} max {TRIP['max_night']}€/noche",
    backstory="Superhost experto en relación calidad-precio. Solo datos reales.",
    tools=[search_accommodation, search_accommodation_image, web_search],
    verbose=True,
    allow_delegation=False,
)

accommodation_result = run_agent_test(
    "Agente de Alojamientos",
    accommodation_agent,
    f"Busca 3 opciones de Airbnb en {TRIP['destination']} para la noche 1.\n"
    f"{TRIP['adults']} huéspedes, máximo {TRIP['max_night']}€/noche.\n\n"
    "INSTRUCCIONES:\n"
    "1. Usa 'Accommodation search' para buscar opciones reales.\n"
    "2. Usa 'Accommodation image search' para cada opción.\n"
    "3. Incluye image_url y listing_url en cada opción.\n\n"
    "FORMATO: JSON con {\"night\": 1, \"city\": ..., \"options\": "
    "[{\"name\": ..., \"price\": ..., \"image_url\": ..., \"listing_url\": ...}]}"
)

---
## 📊 Resumen de Tests

In [ ]:
print("\n" + "="*60)
print("📊 RESUMEN DE TESTS")
print("="*60)

tests = [
    ("✈️ Vuelos", flight_result),
    ("🚗 Transporte", transport_result),
    ("🎯 Actividades", activities_result),
    ("🏠 Alojamientos", accommodation_result),
]

for name, result in tests:
    if isinstance(result, dict):
        print(f"  ✅ {name} — JSON válido ({len(json.dumps(result))} chars)")
    elif isinstance(result, str) and len(result) > 50:
        print(f"  ⚠️ {name} — Respuesta texto ({len(result)} chars)")
    else:
        print(f"  ❌ {name} — Sin resultado")

print("\n" + "="*60)